# LLaMA-3.1-8B BiE Zero-Shot Accuracy

Sweep **BiE8 through BiE4** using the existing G16, independent per-tensor P95–P99 threshold search. Every width reloads the same FP16 checkpoint, calibrates activation thresholds on eight WikiText-2 train blocks, selects weight and activation thresholds by tensor reconstruction MSE, and evaluates the eight zero-shot tasks used by the FP16 Baseline and Vanilla BFP notebooks (LAMBADA, ARC-Easy, ARC-Challenge, PIQA, HellaSwag, WinoGrande, BoolQ, and COPA). Decoder Linear weights and inputs use BiE; `lm_head` remains FP16.

Results use `lm-evaluation-harness==0.4.13`, no few-shot examples, no request cache, complete task datasets, the same paper metric policy, and the pinned LLaMA-2 model revision. Each width writes a JSON immediately. The final cell downloads a ZIP containing all five JSON files. Run this notebook in a fresh Colab GPU runtime.


In [ ]:
%pip install -q "lm_eval[hf]==0.4.13" "transformers==5.16.1" "datasets==4.8.5" accelerate sentencepiece tqdm


In [ ]:
import gc
import importlib.metadata
import json
import math
import os
import platform
import struct
from functools import lru_cache
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import datasets
import lm_eval
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager
from lm_eval.utils import make_table

MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_SLUG = "llama3.1-8b"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CALIBRATION_SPLIT = "train"
CALIBRATION_BLOCKS = 8
CALIBRATION_SEED = 0
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
BASELINE_PPL = 6.237492084503174


@dataclass(frozen=True)
class BiEConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 4  # BiE5 default for synthetic checks; sweep overrides this.
    rounding: str = "nearest_even"
    threshold_method: str = "independent_percentile_grid_reconstruction_mse"
    candidate_percentiles: tuple = tuple(range(95, 100))
    threshold_exponent_bits: int = 64
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 256
    activation_histogram_chunk_rows: int = 512
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This experiment requires Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This experiment requires two E5 shared exponents.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding != "nearest_even":
            raise ValueError("This experiment requires round-to-nearest-even.")
        if self.threshold_method != "independent_percentile_grid_reconstruction_mse":
            raise ValueError("Unexpected threshold method.")
        if self.candidate_percentiles != tuple(range(95, 100)):
            raise ValueError("Expected the inclusive P95-P99 grid at 1-percent steps.")
        if self.threshold_exponent_bits != 64:
            raise ValueError("Threshold log2 values are stored as float64.")
        if min(self.weight_chunk_rows, self.activation_chunk_rows,
               self.activation_histogram_chunk_rows) <= 0:
            raise ValueError("Chunk sizes must be positive.")



BIE = BiEConfig()
BIE.validate()
MANTISSA_BITS_SWEEP = (7, 6, 5, 4, 3)  # BiE8 through BiE4.
TASKS = ["lambada_openai", "arc_easy", "arc_challenge", "piqa", "hellaswag", "winogrande", "boolq", "copa"]
NUM_FEWSHOT = 0
BATCH_SIZE = "auto"
MAX_BATCH_SIZE = 64
BOOTSTRAP_ITERS = 0
LOG_SAMPLES = True
OUTPUT_DIR = Path("result")
ARCHIVE_PATH = Path("llama3.1-8b-bie8-bie4-g16-p95-p99-zero-shot.zip")

EXPECTED_SAMPLES = {
    "lambada_openai": 5153,
    "arc_easy": 2376,
    "arc_challenge": 1172,
    "piqa": 1838,
    "hellaswag": 10042,
    "winogrande": 1267,
    "boolq": 3270,
    "copa": 100,
}
PAPER_METRICS = {
    "lambada_openai": "acc,none",
    "arc_easy": "acc_norm,none",
    "arc_challenge": "acc_norm,none",
    "piqa": "acc_norm,none",
    "hellaswag": "acc_norm,none",
    "winogrande": "acc,none",
    "boolq": "acc,none",
    "copa": "acc,none",
}
DISPLAY_NAMES = {
    "lambada_openai": "LAMBADA",
    "arc_easy": "ARC-Easy",
    "arc_challenge": "ARC-Challenge",
    "piqa": "PIQA",
    "hellaswag": "HellaSwag",
    "winogrande": "WinoGrande",
    "boolq": "BoolQ",
    "copa": "COPA",
}
MODEL_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
# Upload the matching 01_Baseline result JSON next to this notebook to record deltas vs FP16.
FP16_BASELINE_PATH = Path("llama3.1-8b-fp16-zero-shot.json")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")
if importlib.metadata.version("lm_eval") != "0.4.13":
    raise RuntimeError("This experiment requires lm_eval==0.4.13.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"BiE formats: {[f'BiE{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")
print("G16, tensor-wise P95-P99 threshold search, FP64 threshold exponents, RNE")


## BiE quantization and independent threshold search

This section is copied from the P95–P99 BiE PPL sweep: nearest-rank FP16 percentile candidates, full BiE reconstruction MSE, FP64 threshold exponent storage, G16 alignment, two shared E5 exponents, and round-to-nearest-even mantissas.


In [ ]:
COUNT_NAMES = (
    "total_values", "outlier_values", "total_blocks", "outlier_blocks",
    "normal_only_blocks", "outlier_only_blocks", "mixed_blocks",
)


def _empty_counts(device):
    return torch.zeros(len(COUNT_NAMES), dtype=torch.int64, device=device)


@lru_cache(maxsize=None)
def _threshold_value(exponent):
    if not isinstance(exponent, (int, float)) or not math.isfinite(exponent):
        raise ValueError("Threshold log2 exponent must be finite.")
    if exponent == -128.0:
        return 0.0  # Reserved exponent for an all-zero percentile.
    value = math.exp2(float(exponent))
    if value <= 0.0 or value > 65504.0 * (1.0 + 1e-12):
        raise ValueError("Threshold exponent is outside finite FP16 range.")
    return float(struct.unpack("e", struct.pack("e", value))[0])


@torch.no_grad()
def _absolute_fp16_histogram(tensor, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0 or chunk_rows <= 0:
        raise ValueError("Expected a nonempty tensor and positive chunk_rows.")
    histogram = torch.zeros(32768, dtype=torch.int64, device=tensor.device)
    for start in range(0, flat.size(0), chunk_rows):
        values = flat[start:start + chunk_rows].to(torch.float16).contiguous()
        magnitude_bits = values.view(torch.int16).to(torch.int32).bitwise_and(0x7fff)
        histogram.add_(torch.bincount(magnitude_bits.reshape(-1), minlength=32768))
    if histogram[0x7c00:].sum().item():
        raise ValueError("Threshold search requires finite FP16 values.")
    return histogram


def _percentile_exponent_candidates(histogram, config):
    hist = histogram.detach().cpu()
    total = int(hist.sum().item())
    if total == 0:
        raise ValueError("No values were collected for threshold search.")
    if hist[0x7c00:].sum().item():
        raise ValueError("Threshold search requires finite FP16 values.")
    cumulative = hist.cumsum(0)
    by_value = {}
    for percentile in config.candidate_percentiles:
        rank = math.ceil(percentile * total / 100)
        bit_pattern = int(torch.searchsorted(cumulative, rank).item())
        value = float(torch.tensor([bit_pattern], dtype=torch.int16).view(torch.float16)[0].item())
        if value not in by_value:
            exponent = -128.0 if value == 0.0 else math.log2(value)
            if _threshold_value(exponent) != value:
                raise RuntimeError("Threshold exponent failed FP16 round-trip.")
            by_value[value] = {
                "threshold_exponent": exponent,
                "threshold_value": value,
                "percentiles": [],
            }
        by_value[value]["percentiles"].append(percentile)
    return list(by_value.values())


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


@torch.no_grad()
def _quantize_bie_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    # BiE has one type bit per value; no sub-block alignment is applied.
    outlier = valid_blocks & (magnitude > threshold)
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)
    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(~normal_present & outlier_present, outlier_exp, normal_exp)
    outlier_exp = torch.where(~outlier_present & normal_present, normal_exp, outlier_exp)
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(-mantissa_max, mantissa_max)
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)

    real_block = valid_blocks.any(dim=-1, keepdim=True)
    normal_only = real_block & normal_present & ~outlier_present
    outlier_only = real_block & outlier_present & ~normal_present
    mixed = real_block & normal_present & outlier_present
    counts = torch.stack((
        torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
        outlier.sum(dtype=torch.int64), real_block.sum(dtype=torch.int64),
        outlier_present.sum(dtype=torch.int64), normal_only.sum(dtype=torch.int64),
        outlier_only.sum(dtype=torch.int64), mixed.sum(dtype=torch.int64),
    ))
    return dequantized, counts


@torch.no_grad()
def quantize_bie(tensor, config, chunk_rows, threshold_exponent):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0 or chunk_rows <= 0:
        raise ValueError("Expected a nonempty tensor and positive chunk_rows.")
    threshold = _threshold_value(threshold_exponent)
    output = torch.empty_like(flat)
    counts = _empty_counts(flat.device)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        quantized, chunk_counts = _quantize_bie_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        counts.add_(chunk_counts)
    return output.reshape_as(tensor), counts


@torch.no_grad()
def _reconstruction_sse(tensor, config, chunk_rows, threshold_exponent):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    total = torch.zeros((), dtype=torch.float64, device=tensor.device)
    threshold = _threshold_value(threshold_exponent)
    for start in range(0, flat.size(0), chunk_rows):
        original = flat[start:start + chunk_rows]
        quantized, _ = _quantize_bie_rows_with_threshold(original, threshold, config)
        total.add_((original.float() - quantized.float()).square().sum(dtype=torch.float64))
    return total


@torch.no_grad()
def search_weight_threshold(weight, config):
    histogram = _absolute_fp16_histogram(weight, config.weight_chunk_rows)
    candidates = _percentile_exponent_candidates(histogram, config)
    for candidate in candidates:
        sse = _reconstruction_sse(
            weight, config, config.weight_chunk_rows, candidate["threshold_exponent"]
        )
        candidate["mse"] = float((sse / weight.numel()).item())
    best = min(candidates, key=lambda row: (row["mse"], row["threshold_value"]))
    return best["threshold_exponent"], {
        "percentile_grid": list(config.candidate_percentiles),
        "candidate_thresholds": candidates,
        "selected_exponent": best["threshold_exponent"],
        "selected_mse": best["mse"],
        "selection_objective": "tensor_reconstruction_mse",
    }


@torch.no_grad()
def _outliers_per_block_histogram(rows, threshold, config):
    width = rows.shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))
    occupancy = flat.abs().gt(threshold).reshape(-1, config.block_size).sum(dim=-1)
    return torch.bincount(occupancy, minlength=config.block_size + 1)


@torch.no_grad()
def quantize_weight_in_place(weight, config, threshold_exponent):
    counts = _empty_counts(weight.device)
    histogram = torch.zeros(config.block_size + 1, dtype=torch.int64, device=weight.device)
    threshold = _threshold_value(threshold_exponent)
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        histogram.add_(_outliers_per_block_histogram(weight[start:end], threshold, config))
        quantized, chunk_counts = _quantize_bie_rows_with_threshold(
            weight[start:end], threshold, config
        )
        weight[start:end].copy_(quantized)
        counts.add_(chunk_counts)
    return counts, histogram

## Activation calibration and Linear replacement

The unquantized FP16 checkpoint supplies calibration activations. BiE threshold search and quantization are identical to the PPL sweep. Auto-batch detection is excluded from runtime telemetry without changing forward quantization.


In [ ]:
def _rate(numerator, denominator):
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(COUNT_NAMES, values)}


def _summarize_counts(counts):
    return {
        "counts": counts,
        "rates": {
            "outlier_value_rate": _rate(counts["outlier_values"], counts["total_values"]),
            "secondary_exponent_value_rate": _rate(counts["outlier_values"], counts["total_values"]),
            "outlier_block_rate": _rate(counts["outlier_blocks"], counts["total_blocks"]),
            "normal_only_block_rate": _rate(counts["normal_only_blocks"], counts["total_blocks"]),
            "outlier_only_block_rate": _rate(counts["outlier_only_blocks"], counts["total_blocks"]),
            "mixed_block_rate": _rate(counts["mixed_blocks"], counts["total_blocks"]),
        },
    }


def _quantized_linear_modules(model, config):
    return {
        name: module for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
        and (config.quantize_lm_head or name != "lm_head")
    }


@torch.inference_mode()
def _run_calibration_forward(model, batches, label):
    device = next(model.parameters()).device
    for batch in tqdm(batches, desc=label):
        output = model(batch.to(device), use_cache=False)
        del output


@torch.inference_mode()
def collect_activation_histograms(model, batches, config):
    modules = _quantized_linear_modules(model, config)
    histograms = {
        name: torch.zeros(32768, dtype=torch.int64, device=module.weight.device)
        for name, module in modules.items()
    }
    hooks = []

    def make_hook(name):
        def hook(_module, inputs):
            histogram = _absolute_fp16_histogram(
                inputs[0], config.activation_histogram_chunk_rows
            )
            histograms[name].add_(histogram)
        return hook

    try:
        for name, module in modules.items():
            hooks.append(module.register_forward_pre_hook(make_hook(name)))
        _run_calibration_forward(model, batches, "Activation percentile histograms")
    finally:
        for hook in hooks:
            hook.remove()

    for name, histogram in histograms.items():
        if not histogram.sum().item():
            raise RuntimeError(f"No calibration activations for {name}.")
    return histograms


@torch.inference_mode()
def search_activation_thresholds(model, batches, histograms, config):
    modules = _quantized_linear_modules(model, config)
    if set(modules) != set(histograms):
        raise RuntimeError("Activation histogram modules do not match the model.")
    candidates = {
        name: _percentile_exponent_candidates(histograms[name], config)
        for name in modules
    }
    sse = {
        name: torch.zeros(len(candidates[name]), dtype=torch.float64, device=module.weight.device)
        for name, module in modules.items()
    }
    counts = {name: 0 for name in modules}
    hooks = []

    def make_hook(name):
        def hook(_module, inputs):
            x = inputs[0].detach()
            width = x.shape[-1]
            flat = x.reshape(-1, width)
            counts[name] += x.numel()
            for index, candidate in enumerate(candidates[name]):
                error = _reconstruction_sse(
                    flat, config, config.activation_chunk_rows,
                    candidate["threshold_exponent"],
                )
                sse[name][index].add_(error)
        return hook

    try:
        for name, module in modules.items():
            hooks.append(module.register_forward_pre_hook(make_hook(name)))
        _run_calibration_forward(model, batches, "Activation threshold MSE")
    finally:
        for hook in hooks:
            hook.remove()

    selected = {}
    diagnostics = {}
    for name in modules:
        if counts[name] == 0:
            raise RuntimeError(f"No activation MSE samples for {name}.")
        for index, candidate in enumerate(candidates[name]):
            candidate["mse"] = float((sse[name][index] / counts[name]).item())
        best = min(
            candidates[name],
            key=lambda row: (row["mse"], row["threshold_value"]),
        )
        selected[name] = best["threshold_exponent"]
        diagnostics[name] = {
            "calibration_values": counts[name],
            "percentile_grid": list(config.candidate_percentiles),
            "candidate_thresholds": candidates[name],
            "selected_exponent": best["threshold_exponent"],
            "selected_mse": best["mse"],
            "selection_objective": "tensor_reconstruction_mse",
        }
    return selected, diagnostics


class BiELinear(nn.Module):
    def __init__(self, linear, config, weight_exponent, weight_counts, weight_histogram,
                 activation_exponent, weight_search, activation_search):
        super().__init__()
        self.linear = linear
        self.config = config
        self.weight_threshold_exponent = float(weight_exponent)
        self.activation_threshold_exponent = float(activation_exponent)
        self.weight_search = weight_search
        self.activation_search = activation_search
        self.weight_counts = _counts_to_dict(weight_counts)
        self.weight_histogram = [int(v) for v in weight_histogram.cpu().tolist()]
        if sum(self.weight_histogram) != self.weight_counts["total_blocks"]:
            raise RuntimeError("Weight block histogram count mismatch.")
        if sum(k * v for k, v in enumerate(self.weight_histogram)) != self.weight_counts["outlier_values"]:
            raise RuntimeError("Weight outlier histogram count mismatch.")
        device = linear.weight.device
        self.collect_telemetry = True
        self.register_buffer("_activation_counts", _empty_counts(device), persistent=False)
        self.register_buffer(
            "_activation_calls", torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )

    def forward(self, x):
        x_bie, counts = quantize_bie(
            x, self.config, self.config.activation_chunk_rows,
            self.activation_threshold_exponent,
        )
        if self.collect_telemetry:
            self._activation_counts.add_(counts)
            self._activation_calls.add_(1)
        return F.linear(x_bie, self.linear.weight, self.linear.bias).to(torch.float16)

    def export_stats(self):
        return {
            "weight": {
                "threshold_exponent": self.weight_threshold_exponent,
                "threshold_value": _threshold_value(self.weight_threshold_exponent),
                "search": self.weight_search,
                "outliers_per_block_histogram": self.weight_histogram,
                **_summarize_counts(self.weight_counts),
            },
            "activation": {
                "threshold_exponent": self.activation_threshold_exponent,
                "threshold_value": _threshold_value(self.activation_threshold_exponent),
                "search": self.activation_search,
                "inference_calls": int(self._activation_calls.detach().cpu().item()),
                **_summarize_counts(_counts_to_dict(self._activation_counts)),
            },
        }


@torch.no_grad()
def replace_linear_layers(model, config, activation_exponents, activation_search):
    modules = _quantized_linear_modules(model, config)
    if set(modules) != set(activation_exponents) or set(modules) != set(activation_search):
        raise RuntimeError("Activation thresholds must cover every quantized Linear.")
    replaced = {}
    for name, linear in tqdm(modules.items(), desc="Weight threshold MSE and BiE quantization"):
        weight_exponent, weight_search = search_weight_threshold(linear.weight, config)
        weight_counts, weight_histogram = quantize_weight_in_place(
            linear.weight, config, weight_exponent
        )
        wrapper = BiELinear(
            linear, config, weight_exponent, weight_counts, weight_histogram,
            activation_exponents[name], weight_search, activation_search[name],
        )
        parent_name, _, child_name = name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, wrapper)
        replaced[name] = wrapper
    return replaced


def _add_counts(total, current):
    for name in COUNT_NAMES:
        total[name] += current[name]


def _exponent_summary(values):
    return {
        "count": len(values),
        "min": min(values) if values else None,
        "max": max(values) if values else None,
        "mean": sum(values) / len(values) if values else None,
    }


def export_quantization_stats(replaced):
    weight_totals = {name: 0 for name in COUNT_NAMES}
    activation_totals = {name: 0 for name in COUNT_NAMES}
    weight_histogram = [0] * (replaced[next(iter(replaced))].config.block_size + 1)
    weight_exponents = []
    activation_exponents = []
    layers = []
    for name in sorted(replaced):
        layer_stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **layer_stats})
        _add_counts(weight_totals, layer_stats["weight"]["counts"])
        _add_counts(activation_totals, layer_stats["activation"]["counts"])
        weight_histogram = [a + b for a, b in zip(
            weight_histogram, layer_stats["weight"]["outliers_per_block_histogram"]
        )]
        weight_exponents.append(layer_stats["weight"]["threshold_exponent"])
        activation_exponents.append(layer_stats["activation"]["threshold_exponent"])
    return {
        "aggregate": {
            "weight": {
                "threshold_exponent_summary": _exponent_summary(weight_exponents),
                "outliers_per_block_histogram": weight_histogram,
                **_summarize_counts(weight_totals),
            },
            "activation": {
                "threshold_exponent_summary": _exponent_summary(activation_exponents),
                **_summarize_counts(activation_totals),
            },
        },
        "layers": layers,
    }

## Synthetic BiE checks

Run the source notebook's numerical checks before downloading the model.


In [ ]:
_test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_zero = torch.zeros((2, 16), dtype=torch.float32, device=_test_device)
_zero_hist = _absolute_fp16_histogram(_zero, chunk_rows=1)
_zero_candidates = _percentile_exponent_candidates(_zero_hist, BIE)
assert len(_zero_candidates) == 1
assert _zero_candidates[0]["threshold_exponent"] == -128.0
_zero_q, _zero_counts = quantize_bie(_zero, BIE, 1, -128.0)
assert torch.equal(_zero_q, _zero)
assert _counts_to_dict(_zero_counts)["outlier_values"] == 0

# Nearby percentile values must remain distinct thresholds below 2 ** 0.
_dense = torch.linspace(0.6, 0.9, 64, device=_test_device).reshape(4, 16)
_dense_candidates = _percentile_exponent_candidates(
    _absolute_fp16_histogram(_dense, 2), BIE
)
assert 2 <= len(_dense_candidates) <= len(BIE.candidate_percentiles)
assert all(0.6 <= row["threshold_value"] <= 0.91 for row in _dense_candidates)
assert all(row["threshold_value"] < 1.0 for row in _dense_candidates)
assert all(
    _threshold_value(row["threshold_exponent"]) == row["threshold_value"]
    for row in _dense_candidates
)

_mixed = torch.zeros((1, 16), dtype=torch.float32, device=_test_device)
_mixed[0, 0] = 0.9
_mixed[0, 1] = -0.4
_mixed[0, 2] = 4.5
_mixed[0, 3] = 0.84375
_mixed_q, _mixed_counts = quantize_bie(_mixed, BIE, 1, 0)
_mixed_stats = _counts_to_dict(_mixed_counts)
assert _mixed_stats["outlier_values"] == 1
assert _mixed_stats["mixed_blocks"] == 1
assert _mixed_q[0, 0].item() == 0.875
assert _mixed_q[0, 1].item() == -0.375
assert _mixed_q[0, 2].item() == 4.5
assert _mixed_q[0, 3].item() == 0.875  # 13.5 rounds to even 14, not truncation 13.

_all_outlier = torch.ones((1, 16), dtype=torch.float32, device=_test_device)
_, _all_outlier_counts = quantize_bie(_all_outlier, BIE, 1, -1)
assert _counts_to_dict(_all_outlier_counts)["outlier_only_blocks"] == 1

torch.manual_seed(7)
_chunk_input = torch.randn((5, 32), dtype=torch.float32, device=_test_device)
_chunk_q1, _chunk_c1 = quantize_bie(_chunk_input, BIE, 1, 0)
_chunk_q5, _chunk_c5 = quantize_bie(_chunk_input, BIE, 5, 0)
assert torch.equal(_chunk_q1, _chunk_q5)
assert torch.equal(_chunk_c1, _chunk_c5)

_weight = torch.linspace(-1.2, 1.2, 64, device=_test_device).reshape(4, 16)
_weight_exp, _weight_search = search_weight_threshold(_weight, BIE)
assert len(_weight_search["candidate_thresholds"]) >= 2
assert _weight_exp == min(
    _weight_search["candidate_thresholds"],
    key=lambda row: (row["mse"], row["threshold_value"]),
)["threshold_exponent"]
_weight_q, _ = quantize_bie(_weight, BIE, 2, _weight_exp)
assert torch.allclose(
    (_weight.float() - _weight_q.float()).square().mean(),
    torch.tensor(_weight_search["selected_mse"], device=_test_device),
    rtol=1e-6, atol=1e-8,
)

class _ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(16, 8, bias=False, device=_test_device)
        self.proj2 = nn.Linear(16, 8, bias=False, device=_test_device)
        with torch.no_grad():
            self.proj2.weight.copy_(self.proj.weight * 16)

    def forward(self, x, use_cache=False):
        return self.proj(x) + self.proj2(x * 8)


_toy = _ToyModel().eval()
_calib_batches = [
    torch.linspace(-2, 2, 32, device=_test_device).reshape(2, 16),
    torch.linspace(-0.5, 0.5, 48, device=_test_device).reshape(3, 16),
]
_activation_hist = collect_activation_histograms(_toy, _calib_batches, BIE)
_activation_exp, _activation_search = search_activation_thresholds(
    _toy, _calib_batches, _activation_hist, BIE
)
assert set(_activation_exp) == {"proj", "proj2"}
assert _activation_search["proj"]["calibration_values"] == 80
assert len(_activation_search["proj"]["candidate_thresholds"]) >= 2
assert math.isclose(_activation_exp["proj2"], _activation_exp["proj"] + 3, abs_tol=1e-12)
_raw_activation = torch.cat(_calib_batches, dim=0)
for _candidate in _activation_search["proj"]["candidate_thresholds"]:
    _q, _ = quantize_bie(
        _raw_activation, BIE, 1, _candidate["threshold_exponent"]
    )
    _mse = (_raw_activation - _q).square().mean().item()
    assert math.isclose(_candidate["mse"], _mse, rel_tol=1e-5, abs_tol=1e-8)
_toy_layers = replace_linear_layers(_toy, BIE, _activation_exp, _activation_search)
assert len(_toy_layers) == 2
assert (
    math.isclose(
        _toy_layers["proj2"].weight_threshold_exponent,
        _toy_layers["proj"].weight_threshold_exponent + 4, abs_tol=1e-12
    )
)
_toy_y = _toy(_calib_batches[0])
_toy_stats = _toy_layers["proj"].export_stats()
assert _toy_y.shape == (2, 8)
assert torch.isfinite(_toy_y).all()
assert sum(_toy_stats["weight"]["outliers_per_block_histogram"]) == _toy_stats["weight"]["counts"]["total_blocks"]
assert _toy_stats["activation"]["inference_calls"] == 1
_toy_aggregate = export_quantization_stats(_toy_layers)["aggregate"]["weight"]
_toy_histogram = _toy_aggregate["outliers_per_block_histogram"]
assert len(_toy_histogram) == BIE.block_size + 1
assert sum(_toy_histogram) == _toy_aggregate["counts"]["total_blocks"]
assert sum(k * v for k, v in enumerate(_toy_histogram)) == _toy_aggregate["counts"]["outlier_values"]
assert _toy_stats["activation"]["threshold_exponent"] == _activation_exp["proj"]
assert _toy_layers["proj2"].export_stats()["activation"]["inference_calls"] == 1
_saved = json.loads(json.dumps(_toy_stats))
assert _saved["weight"]["threshold_exponent"] == _toy_stats["weight"]["threshold_exponent"]
assert _saved["activation"]["threshold_exponent"] == _activation_exp["proj"]

del _zero, _dense, _mixed, _all_outlier, _chunk_input, _weight, _toy, _toy_y
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
for _private_bits in range(4, 9):
    _config = replace(BIE, mantissa_bits=_private_bits - 1)
    _config.validate()
    _check_rows = torch.tensor([[0.25, -0.4, 0.9, 4.5] + [0.0] * 12], device=_test_device)
    _check_q, _check_counts = quantize_bie(_check_rows, _config, 1, 0.0)
    assert _check_q.shape == _check_rows.shape
    assert _counts_to_dict(_check_counts)["outlier_values"] == 1
    _check_exp, _check_search = search_weight_threshold(_check_rows, _config)
    assert _check_exp == min(
        _check_search["candidate_thresholds"],
        key=lambda row: (row["mse"], row["threshold_value"]),
    )["threshold_exponent"]
print("Synthetic BiE4–BiE8 threshold-search checks passed.")

## Validate the zero-shot protocol

Check task names, expected sample counts, and the pinned model revision against the FP16 Baseline and Vanilla BFP experiments. Upload the matching `01_Baseline` result JSON (`FP16_BASELINE_PATH`) to the Colab working directory to record deltas vs FP16; the notebook checks that its model, revision, lm-eval version, tasks, metrics, and example count match. Without it, deltas are stored as null.


In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")

task_manager = TaskManager()
matched_tasks = task_manager.match_tasks(TASKS)
missing_tasks = sorted(set(TASKS) - set(matched_tasks))
if missing_tasks:
    raise RuntimeError(f"Tasks missing from lm-eval: {missing_tasks}")

resolved_model_revision = model_info(MODEL_ID, token=token).sha
if resolved_model_revision != MODEL_REVISION:
    raise RuntimeError(
        "Model revision differs from the pinned revision: "
        f"{resolved_model_revision} != {MODEL_REVISION}"
    )
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=resolved_model_revision, token=token
)
print(f"lm-eval: {importlib.metadata.version('lm_eval')}")
print(f"Model revision: {resolved_model_revision}")
print(f"Tasks: {', '.join(TASKS)}")
print(f"Examples per format: {sum(EXPECTED_SAMPLES.values()):,}")
print(f"Formats: {[f'BiE{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")


def load_fp16_baseline(path):
    if not path.is_file():
        print(f"FP16 baseline {path} not found; deltas vs FP16 will be stored as null.")
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    checks = {
        "model": (data.get("model"), MODEL_ID),
        "model_revision": (data.get("model_revision"), MODEL_REVISION),
        "num_fewshot": (data.get("num_fewshot"), NUM_FEWSHOT),
        "lm_eval": (data.get("environment", {}).get("lm_eval"), importlib.metadata.version("lm_eval")),
        "tasks": (list(data.get("paper_scores", {})), TASKS),
        "metrics": (
            {task: row["metric"] for task, row in data.get("paper_scores", {}).items()},
            PAPER_METRICS,
        ),
        "total_evaluated_examples": (
            data.get("total_evaluated_examples"), sum(EXPECTED_SAMPLES.values())
        ),
    }
    for name, (found, expected) in checks.items():
        if found != expected:
            raise RuntimeError(f"FP16 baseline {name} mismatch: {found!r} != {expected!r}")
    scores = {task: float(data["paper_scores"][task]["score"]) for task in TASKS}
    return {
        "source": path.name,
        "created_at_utc": data.get("created_at_utc"),
        "model_revision": data["model_revision"],
        "lm_eval": data["environment"]["lm_eval"],
        "num_fewshot": data["num_fewshot"],
        "total_evaluated_examples": data["total_evaluated_examples"],
        "scores": scores,
        "macro_average": sum(scores.values()) / len(TASKS),
    }


def format_macro_delta(result):
    delta = result["macro_delta_percentage_points_vs_fp16"]
    return "n/a (no FP16 baseline JSON)" if delta is None else f"{delta:+.2f} pp"


BASELINE_REFERENCE = load_fp16_baseline(FP16_BASELINE_PATH)
if BASELINE_REFERENCE is not None:
    print(
        f"FP16 baseline: {BASELINE_REFERENCE['source']} "
        f"(macro average {BASELINE_REFERENCE['macro_average'] * 100:.2f}%)"
    )


## WikiText-2 train calibration

Use the same eight seeded, non-overlapping 2048-token train blocks as the BiE PPL sweep. The zero-shot evaluation tasks do not enter threshold selection.


In [ ]:
calibration_dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=CALIBRATION_SPLIT)
calibration_text = "\n\n".join(calibration_dataset["text"])
calibration_ids = tokenizer(calibration_text, return_tensors="pt").input_ids
calibration_source_tokens = calibration_ids.numel()
available_calibration_blocks = calibration_source_tokens // CONTEXT_LENGTH
if available_calibration_blocks < CALIBRATION_BLOCKS:
    raise RuntimeError("Not enough full WikiText-2 train blocks for calibration.")
generator = torch.Generator().manual_seed(CALIBRATION_SEED)
calibration_block_indices = sorted(
    torch.randperm(available_calibration_blocks, generator=generator)[:CALIBRATION_BLOCKS].tolist()
)
calibration_batches = [
    calibration_ids[:, index * CONTEXT_LENGTH:(index + 1) * CONTEXT_LENGTH].clone()
    for index in calibration_block_indices
]
del calibration_dataset, calibration_text, calibration_ids
print(f"Calibration: {len(calibration_batches)} WikiText-2 train blocks, seed {CALIBRATION_SEED}")
print(f"Sampled block indices: {calibration_block_indices}")


## Load and quantize each width

For every format, reload the pinned FP16 checkpoint, calibrate activation thresholds, then search and quantize all 224 decoder Linear layers.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("BiE calibration and zero-shot evaluation require an NVIDIA CUDA GPU.")

def load_and_quantize_model(config):
    if not torch.cuda.is_available():
        raise RuntimeError("BiE calibration and zero-shot evaluation require an NVIDIA CUDA GPU.")

    torch.manual_seed(0)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=resolved_model_revision,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False

    original_linear_modules = _quantized_linear_modules(model, config)
    if len(original_linear_modules) != 224:
        raise RuntimeError(
            f"Expected 224 decoder Linear layers, got {len(original_linear_modules)}."
        )

    activation_histograms = collect_activation_histograms(model, calibration_batches, config)
    activation_exponents, activation_search = search_activation_thresholds(
        model, calibration_batches, activation_histograms, config
    )
    del activation_histograms
    torch.cuda.empty_cache()

    bie_layers = replace_linear_layers(model, config, activation_exponents, activation_search)
    if set(bie_layers) != set(original_linear_modules):
        raise RuntimeError("Quantized decoder Linear layer set changed.")
    if "lm_head" in bie_layers or not isinstance(model.lm_head, nn.Linear):
        raise RuntimeError("lm_head must remain an FP16 nn.Linear.")
    del original_linear_modules, activation_exponents, activation_search
    torch.cuda.empty_cache()

    parameter_dtypes = {
        parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()
    }
    parameter_devices = {parameter.device.type for parameter in model.parameters()}
    assert parameter_dtypes == {torch.float16}, parameter_dtypes
    assert parameter_devices == {"cuda"}, parameter_devices

    print(f"BiE{1 + config.mantissa_bits}-quantized Linear layers: {len(bie_layers)}")
    print("lm_head: FP16; threshold: per-tensor P95-P99 exponent grid, reconstruction MSE")
    print("Format: G16, per-value type bit, two E5 shared exponents, round-to-nearest-even mantissa")
    return model, bie_layers



class BiEEvaluationHFLM(HFLM):
    def __init__(self, *args, bie_layers, **kwargs):
        self.bie_layers = bie_layers
        super().__init__(*args, **kwargs)

    def _detect_batch_size(self, requests=None, pos=0):
        for layer in self.bie_layers.values():
            layer.collect_telemetry = False
        try:
            return super()._detect_batch_size(requests=requests, pos=pos)
        finally:
            for layer in self.bie_layers.values():
                layer.collect_telemetry = True


## Summarize task scores and validate complete evaluation

Use the same eight paper metrics, sample-count closure, and FP16 reference as the BFP zero-shot notebook.


In [ ]:
def to_jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def summarize_evaluation(raw_results):
    closure = {}
    for task_name, expected in EXPECTED_SAMPLES.items():
        sample_info = raw_results["n-samples"][task_name]
        original = int(sample_info["original"])
        effective = int(sample_info["effective"])
        passed = original == expected and effective == expected
        closure[task_name] = {
            "expected": expected,
            "original": original,
            "effective": effective,
            "passed": passed,
        }
        if not passed:
            raise RuntimeError(
                f"Evaluation closure failed for {task_name}: "
                f"expected={expected}, original={original}, effective={effective}"
            )

    paper_scores = {}
    for task_name in TASKS:
        metric_name = PAPER_METRICS[task_name]
        metrics = raw_results["results"][task_name]
        if metric_name not in metrics:
            raise KeyError(
                f"Missing {metric_name!r} for {task_name}; available: {sorted(metrics)}"
            )
        score = float(metrics[metric_name])
        baseline_score = (
            None if BASELINE_REFERENCE is None
            else BASELINE_REFERENCE["scores"][task_name]
        )
        stderr_name = metric_name.replace(",none", "_stderr,none")
        stderr_raw = metrics.get(stderr_name)
        try:
            stderr = float(stderr_raw)
        except (TypeError, ValueError):
            stderr = None
        paper_scores[task_name] = {
            "display_name": DISPLAY_NAMES[task_name],
            "metric": metric_name,
            "score": score,
            "score_percent": score * 100.0,
            "fp16_baseline_score": baseline_score,
            "fp16_baseline_score_percent": None if baseline_score is None else baseline_score * 100.0,
            "delta_vs_fp16": None if baseline_score is None else score - baseline_score,
            "delta_percentage_points_vs_fp16": (
                None if baseline_score is None else (score - baseline_score) * 100.0
            ),
            "stderr": stderr,
            "stderr_percent": None if stderr is None else stderr * 100.0,
        }

    macro_average = sum(x["score"] for x in paper_scores.values()) / len(TASKS)
    return closure, paper_scores, macro_average


## Sweep BiE8 through BiE4

Each width saves a complete JSON with task scores, deltas from FP16, threshold exponents, calibration metadata, quantization statistics, and the raw lm-eval output. Completed widths stay on disk if a later run stops.


In [ ]:
results = []
output_paths = []

for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(BIE, mantissa_bits=mantissa_bits)
    config.validate()
    bie_bits = 1 + config.mantissa_bits
    output_path = OUTPUT_DIR / (
        f"{MODEL_SLUG}-bie{bie_bits}-g16-p95-p99-grid-mse-rne-fp64exp-"
        "calib-train-no-lm-head-s2048-zero-shot.json"
    )
    print(f"\n{'=' * 72}\nRunning BiE{bie_bits}: {output_path.name}\n{'=' * 72}")

    model, layers = load_and_quantize_model(config)
    try:
        eval_model = BiEEvaluationHFLM(
            pretrained=model,
            tokenizer=tokenizer,
            batch_size=BATCH_SIZE,
            max_batch_size=MAX_BATCH_SIZE,
            bie_layers=layers,
        )
        evaluation_start = time.perf_counter()
        raw_results = lm_eval.simple_evaluate(
            model=eval_model,
            tasks=TASKS,
            num_fewshot=NUM_FEWSHOT,
            batch_size=BATCH_SIZE,
            max_batch_size=MAX_BATCH_SIZE,
            device="cuda:0",
            limit=None,
            use_cache=None,
            bootstrap_iters=BOOTSTRAP_ITERS,
            check_integrity=False,
            log_samples=LOG_SAMPLES,
            task_manager=task_manager,
            random_seed=0,
            numpy_random_seed=1234,
            torch_random_seed=1234,
            fewshot_random_seed=1234,
        )
        evaluation_elapsed_seconds = time.perf_counter() - evaluation_start
        if raw_results is None:
            raise RuntimeError("lm-eval returned no results.")
        closure, paper_scores, macro_average = summarize_evaluation(raw_results)
        quantization_stats = export_quantization_stats(layers)
        if len(quantization_stats["layers"]) != 224:
            raise RuntimeError("BiE layer metadata count mismatch.")
        if any(
            row["activation"]["inference_calls"] == 0
            for row in quantization_stats["layers"]
        ):
            raise RuntimeError("A BiE layer recorded no official evaluation forwards.")
        for domain in ("weight", "activation"):
            counts = quantization_stats["aggregate"][domain]["counts"]
            if counts["total_blocks"] != (
                counts["normal_only_blocks"]
                + counts["outlier_only_blocks"]
                + counts["mixed_blocks"]
            ):
                raise RuntimeError(f"{domain} block counts do not close.")
            if counts["outlier_values"] > counts["total_values"]:
                raise RuntimeError(f"{domain} outlier count exceeds total values.")
        weight_histogram = quantization_stats["aggregate"]["weight"]["outliers_per_block_histogram"]
        weight_counts = quantization_stats["aggregate"]["weight"]["counts"]
        if (sum(weight_histogram) != weight_counts["total_blocks"] or
            sum(k * n for k, n in enumerate(weight_histogram)) != weight_counts["outlier_values"]):
            raise RuntimeError("Weight outliers-per-block histogram does not close.")

        threshold_exponents = {
            "weight": {name: layer.weight_threshold_exponent for name, layer in sorted(layers.items())},
            "activation": {name: layer.activation_threshold_exponent for name, layer in sorted(layers.items())},
        }
        if any(len(mapping) != 224 for mapping in threshold_exponents.values()):
            raise RuntimeError("Incomplete tensor-wise threshold exponent map.")
        raw_jsonable = json.loads(json.dumps(raw_results, ensure_ascii=False, default=to_jsonable))
        effective_bits_per_value = (
            bie_bits + 1 + 2 * config.shared_exponent_bits / config.block_size
        )
        macro_delta = (
            None if BASELINE_REFERENCE is None
            else macro_average - BASELINE_REFERENCE["macro_average"]
        )
        result = {
            "schema_version": 1,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "experiment": "bie_zero_shot_accuracy",
            "model": MODEL_ID,
            "model_revision": resolved_model_revision,
            "dtype": "float16",
            "quantized": True,
            "quantization": "decoder Linear W/A BiE fake quantization; FP16 lm_head",
            "format": f"BiE{bie_bits} (1S{config.mantissa_bits}M + two shared E{config.shared_exponent_bits} + type bit, G{config.block_size})",
            "bie_config": asdict(config),
            "sweep_mantissa_bits": list(MANTISSA_BITS_SWEEP),
            "threshold_contract": {
                "candidate_distribution": "absolute FP16 tensor values",
                "percentile_method": "nearest_rank",
                "percentiles": list(config.candidate_percentiles),
                "exponent_mapping": "float64 log2(exact FP16 percentile value); zero uses -128.0 sentinel",
                "candidate_deduplication": "identical FP16 percentile values only",
                "comparison": "outlier iff abs(x) > FP16_round(2 ** threshold_exponent)",
                "selection": "independent argmin tensor reconstruction MSE",
                "weight_granularity": "one complete nn.Linear weight tensor",
                "activation_granularity": "one nn.Linear input tensor across all calibration calls",
                "activation_threshold_frozen_for_inference": True,
                "threshold_storage": "float64 log2 exponent per weight/activation tensor",
            },
            "calibration": {
                "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
                "split": CALIBRATION_SPLIT,
                "source_tokens": calibration_source_tokens,
                "context_length": CONTEXT_LENGTH,
                "sampled_blocks": CALIBRATION_BLOCKS,
                "sampled_tokens": CALIBRATION_BLOCKS * CONTEXT_LENGTH,
                "seed": CALIBRATION_SEED,
                "block_indices": calibration_block_indices,
                "activation_source": "unquantized FP16 model Linear inputs",
            },
            "storage_contract": {
                "private_value_bits": bie_bits,
                "type_bits_per_value": 1,
                "shared_exponents_per_block": 2,
                "shared_exponent_bits": config.shared_exponent_bits,
                "effective_bits_per_value_excluding_tensor_threshold": effective_bits_per_value,
                "mantissa_rounding_contract": "round-to-nearest-even (torch.round), then symmetric saturation",
                "threshold_exponent_bits_per_weight_tensor": config.threshold_exponent_bits,
                "threshold_exponent_bits_per_activation_tensor": config.threshold_exponent_bits,
                "packed_storage_implemented": False,
            },
            "quantized_scope": {
                "operations": "all decoder nn.Linear modules",
                "weights": True, "activations": True, "lm_head": False,
                "attention_internal_matmul": False, "softmax": False, "layernorm": False,
            },
            "linear_output_dtype": "float16",
            "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
            "quantized_linear_layers": len(layers),
            "attention_implementation": "eager",
            "tasks": TASKS,
            "num_fewshot": NUM_FEWSHOT,
            "limit": None,
            "batch_size": BATCH_SIZE,
            "max_batch_size": MAX_BATCH_SIZE,
            "bootstrap_iters": BOOTSTRAP_ITERS,
            "log_samples": LOG_SAMPLES,
            "request_cache_enabled": False,
            "paper_metric_policy": {
                "lambada": "acc", "arc_easy_arc_challenge_piqa_hellaswag": "acc_norm", "winogrande_boolq_copa": "acc",
            },
            "paper_scores": paper_scores,
            "macro_average": macro_average,
            "macro_average_percent": macro_average * 100.0,
            "fp16_baseline_reference": BASELINE_REFERENCE,
            "macro_delta_vs_fp16": macro_delta,
            "macro_delta_percentage_points_vs_fp16": None if macro_delta is None else macro_delta * 100.0,
            "evaluation_closure": closure,
            "total_evaluated_examples": sum(row["effective"] for row in closure.values()),
            "evaluation_elapsed_seconds": evaluation_elapsed_seconds,
            "threshold_exponents": threshold_exponents,
            "quantization_stats": quantization_stats,
            "validation": {
                "expected_quantized_linear_layers": 224,
                "expected_evaluation_counts": EXPECTED_SAMPLES,
                "evaluation_closure_passed": True,
                "weight_histogram_closure_passed": True,
                "auto_batch_probe_telemetry_excluded": True,
            },
            "environment": {
                "gpu": torch.cuda.get_device_name(0),
                "cuda": torch.version.cuda,
                "python": platform.python_version(),
                "pytorch": torch.__version__,
                "transformers": transformers.__version__,
                "datasets": datasets.__version__,
                "lm_eval": importlib.metadata.version("lm_eval"),
            },
            "lm_eval_raw": raw_jsonable,
        }
        output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
        results.append(result)
        output_paths.append(output_path)
        print(make_table(raw_results))
        print(f"BiE{bie_bits} macro average: {result['macro_average_percent']:.2f}%")
        print(f"Delta vs FP16: {format_macro_delta(result)}")
        print(f"Saved: {output_path.resolve()}")
    finally:
        if "eval_model" in locals():
            del eval_model
        del layers, model
        gc.collect()
        torch.cuda.empty_cache()

if len(output_paths) != len(MANTISSA_BITS_SWEEP):
    raise RuntimeError("BiE8-BiE4 sweep did not produce all five result JSON files.")
with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for output_path in output_paths:
        archive.write(output_path, arcname=output_path.name)
print("\nSweep complete:")
for result in results:
    print(f"{result['format'].split()[0]}: average={result['macro_average_percent']:.2f}%, "
          f"delta={format_macro_delta(result)}")
print(f"Archive: {ARCHIVE_PATH.resolve()}")


## Download all five JSON results

Run after the sweep completes. The ZIP contains BiE8, BiE7, BiE6, BiE5, and BiE4 result JSON files.


In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))
